# Task 3 — Cognitive Load and the Data-Ink Ratio

**Data:** `car_crashes.csv` (51 rows, 50 states + DC) and `tips.csv` (244 rows).

This task measures how much **extraneous cognitive load** a chart can carry, then strips it away one step at a time. The goal is to demonstrate that removing non-data ink reduces the work the reader must do — without removing the data itself.

**Three types of cognitive load (Sweller):**
- **Intrinsic** — difficulty inherent in the data itself. Cannot be removed; can be sequenced.
- **Extraneous** — load added by the design. Chart junk, legends, decoding work. *This is your job to remove.*
- **Germane** — effort that goes into actually understanding. Protect it — it is the point.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

crashes = pd.read_csv('data/car_crashes.csv')
tips = pd.read_csv('data/tips.csv')
print('car_crashes shape:', crashes.shape)
print('columns:', list(crashes.columns))
crashes.head()

In [ ]:
# ── Step 1: BAD EXAMPLE — pile on every source of extraneous load ─────────
crashes_sorted_bad = crashes.sort_values('total')  # keep original order for the bad chart
crashes_orig = crashes.copy()  # save for later use with sorted version

cmap_colors = plt.cm.tab20.colors
bar_colors = [cmap_colors[i % len(cmap_colors)] for i in range(51)]

fig, ax = plt.subplots(figsize=(20, 8))
bars = ax.bar(range(51), crashes_orig['total'],
              color=bar_colors, edgecolor='black', linewidth=0.8)

# Every bar a different colour, 51-entry legend
patches = [mpatches.Patch(color=bar_colors[i], label=crashes_orig.iloc[i]['abbrev'])
           for i in range(51)]
ax.legend(handles=patches, ncol=5, fontsize=6,
          loc='upper left', frameon=True, edgecolor='black')

# Rotated labels, heavy gridlines, box frame
ax.set_xticks(range(51))
ax.set_xticklabels(crashes_orig['abbrev'], rotation=90, fontsize=7)
ax.grid(True, which='both', linewidth=1.0, color='black', alpha=0.5)
ax.yaxis.grid(True, linewidth=1.0, color='black', alpha=0.5)
for spine in ax.spines.values():
    spine.set_linewidth(2)
    spine.set_visible(True)

ax.set_xlabel('State (x axis)')
ax.set_ylabel('Total crashes (y axis)')
ax.set_title('BAD EXAMPLE: Total fatal car crashes per state — x axis vs y axis', fontsize=14)
ax.set_ylim(0)

plt.tight_layout()
plt.savefig('bad_example_crashes.png', dpi=300, bbox_inches='tight')
plt.show()

## Step 2 — Count the load

**Load classification of every element in the BAD EXAMPLE:**

| Element | Type | Reason |
|---|---|---|
| Bar height (encodes `total` per state) | **Intrinsic** | This is the data — cannot be removed |
| 51 bars | **Intrinsic** | One per state; the density is inherent |
| 51 different bar colours | **Extraneous** | Colour carries no information here; each state is already identified by its label |
| 51-entry legend | **Extraneous** | Redundant with the x-axis labels; forces 51 memory look-ups |
| Rotated 90° state labels | **Extraneous** | Harder to read than 45° or horizontal; rotation is decorative |
| Heavy horizontal + vertical gridlines | **Extraneous** | Both directions add ink; horizontal-only is enough for bar charts |
| Thick box frame on all 4 sides | **Extraneous** | Adds ink without aiding comparison; only the x baseline is needed |
| Title naming the axes ('x axis vs y axis') | **Extraneous** | Does not state a finding; forces the reader to synthesise meaning |
| Black bar edges | **Extraneous** | Adds noise; bars separate naturally by position |

**Data-ink ratio estimate:**

- Ink that encodes a number = 51 bars × bar height ≈ **51 data marks**
- Total ink ≈ 51 bars + 51 legend patches + 51 colour patches + gridlines (≈ 100 lines) + box frame + labels
  - Rough count: 51 + 51 + 51 + 100 + 4 sides + ~300 label characters ≈ **557 non-trivial ink marks**
- Data-ink ratio ≈ 51 / (51 + 51 + 51 + 100 + 4) ≈ **51 / 257 ≈ 0.20**

Only about one-fifth of the ink is doing work. The rest is extraneous load.

In [ ]:
# ── Step 3: Five-panel progressive de-cluttering ──────────────────────────
crashes_s = crashes.sort_values('total').reset_index(drop=True)
x = range(len(crashes_s))
GREY = '#bbbbbb'
BLUE = '#4477aa'

fig, axes = plt.subplots(1, 5, figsize=(25, 6))

# Panel 0: Remove the legend only
ax = axes[0]
cmap_c = plt.cm.tab20.colors
cols0 = [cmap_c[i % len(cmap_c)] for i in range(51)]
ax.bar(x, crashes_s['total'], color=cols0, edgecolor='black', linewidth=0.5)
ax.set_xticks([])
ax.set_title('Step 1: Remove the legend\n(Extraneous — redundant with x labels)', fontsize=8)
ax.set_ylim(0)

# Panel 1: Remove frame and vertical gridlines
ax = axes[1]
ax.bar(x, crashes_s['total'], color=cols0, edgecolor='black', linewidth=0.5)
ax.yaxis.grid(True, linewidth=0.4, color='#dddddd')
ax.set_xticks([])
for side in ('top', 'right', 'bottom'):
    ax.spines[side].set_visible(False)
ax.spines['left'].set_visible(False)
ax.set_title('Step 2: Remove frame + vertical grid\n(Extraneous — bar height readable from grid alone)', fontsize=8)
ax.set_ylim(0)

# Panel 2: Drop to a single hue
ax = axes[2]
ax.bar(x, crashes_s['total'], color=GREY, edgecolor='none')
ax.yaxis.grid(True, linewidth=0.4, color='#dddddd')
ax.set_xticks([])
for side in ('top', 'right', 'left'):
    ax.spines[side].set_visible(False)
ax.spines['bottom'].set_linewidth(0.5)
ax.set_title('Step 3: Single hue (grey)\n(Extraneous — colour was carrying no information)', fontsize=8)
ax.set_ylim(0)

# Panel 3: Sort bars (already sorted — this is the biggest single edit)
ax = axes[3]
ax.bar(x, crashes_s['total'], color=GREY, edgecolor='none')
ax.yaxis.grid(True, linewidth=0.4, color='#dddddd')
ax.set_xticks([])
for side in ('top', 'right', 'left'):
    ax.spines[side].set_visible(False)
ax.spines['bottom'].set_linewidth(0.5)
ax.set_title('Step 4: Sort bars (already done)\n(Germane — sorting reveals the distribution at no ink cost)', fontsize=8)
ax.set_ylim(0)

# Panel 4: Label only what matters (top 5 + bottom 1)
ax = axes[4]
highlight = crashes_s.nlargest(3, 'total').index.tolist()
colors5 = [BLUE if i in highlight else GREY for i in range(len(crashes_s))]
ax.bar(x, crashes_s['total'], color=colors5, edgecolor='none')
ax.yaxis.grid(True, linewidth=0.4, color='#dddddd')
for i in highlight:
    ax.text(i, crashes_s.iloc[i]['total'] + 0.3,
            crashes_s.iloc[i]['abbrev'], ha='center', va='bottom', fontsize=7,
            color=BLUE, fontweight='bold')
ax.set_xticks([])
for side in ('top', 'right', 'left'):
    ax.spines[side].set_visible(False)
ax.spines['bottom'].set_linewidth(0.5)
ax.set_title('Step 5: Label only top 3\nContext grey, message coloured — one pop-out', fontsize=8)
ax.set_ylim(0)

fig.suptitle('Progressive de-cluttering: same 51 states, progressively less extraneous load',
             x=0.01, ha='left', fontsize=11)
plt.tight_layout()
plt.savefig('deClutter_steps.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ── Step 4: Respect the 4 ± 1 limit — chunk 51 states into 5 regions ──────
# Chunking: US Census regions (4 + DC)
region_map = {
    'Northeast': ['ME','NH','VT','MA','RI','CT','NY','NJ','PA'],
    'Midwest':   ['OH','IN','IL','MI','WI','MN','IA','MO','ND','SD','NE','KS'],
    'South':     ['DE','MD','DC','VA','WV','NC','SC','GA','FL','KY','TN','AL','MS',
                  'AR','LA','OK','TX'],
    'West':      ['MT','ID','WY','CO','NM','AZ','UT','NV','WA','OR','CA','AK','HI'],
}
def get_region(abbrev):
    for reg, states in region_map.items():
        if abbrev in states:
            return reg
    return 'Other'

crashes['region'] = crashes['abbrev'].apply(get_region)
region_mean = crashes.groupby('region')['total'].mean().sort_values(ascending=False)

PALETTE = {'South': '#ee6677', 'West': '#4477aa', 'Midwest': '#228833', 'Northeast': '#ccbb44'}

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(region_mean.index,
              region_mean.values,
              color=[PALETTE.get(r, GREY) for r in region_mean.index],
              edgecolor='none', width=0.6)

for bar, val in zip(bars, region_mean.values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.2,
            f'{val:.1f}', ha='center', va='bottom', fontsize=10)

ax.set_ylabel('Mean total crashes per 100M miles (per state)')
ax.set_title('South has the highest mean fatal crash rate across its states\n'
             '(n=51 states, grouped into 4 Census regions; reader holds ≤ 4 chunks)',
             loc='left')
ax.set_ylim(0)
for side in ('top', 'right'):
    ax.spines[side].set_visible(False)

plt.tight_layout()
plt.savefig('region_chunked.png', dpi=300, bbox_inches='tight')
plt.show()

print('\nChunking choice: 4 US Census regions.')
print('Cost: within-region variation is hidden. SC and FL may differ greatly from the')
print('South mean, but aggregation makes them invisible. Any state-level story is gone.')

In [ ]:
# ── Step 5: BAD EXAMPLE — over-stripped ───────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
crashes_s2 = crashes.sort_values('total').reset_index(drop=True)
ax.bar(range(len(crashes_s2)), crashes_s2['total'], color=GREY, edgecolor='none')
ax.set_xticks([])
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)
ax.set_title('BAD EXAMPLE: no axis labels, no tick values, no units — stripped too far',
             loc='left', fontsize=10)

plt.tight_layout()
plt.savefig('over_stripped.png', dpi=300, bbox_inches='tight')
plt.show()

print("""
Why maximising data-ink blindly fails:
Removing axis labels and tick values deleted the scale. The reader can see
that one bar is taller than another, but cannot say by how much, in what
unit, or which state it is. The bars are now aesthetically minimal but
informationally worthless.

Stopping rule for a colleague in one sentence:
  'Remove ink that does not carry a number or label something a reader
   needs to answer your title question; stop before removing ink that
   does carry a number or names something essential.'
""")

In [ ]:
# ── Key question: adding ink can LOWER cognitive load ──────────────────────
# tips.csv: total bill by day — adding a reference line for the grand mean
# saves the reader from computing the mean themselves.

day_mean = tips.groupby('day')['total_bill'].mean().reindex(['Thur','Fri','Sat','Sun'])
grand_mean = tips['total_bill'].mean()

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(day_mean.index, day_mean.values, color=GREY, edgecolor='none', width=0.5)

# Adding ink: reference line + annotation
ax.axhline(grand_mean, color='#ee6677', linewidth=1.5, linestyle='--', label=f'Grand mean ${grand_mean:.2f}')
ax.text(3.3, grand_mean + 0.3, f'Overall mean\n${grand_mean:.2f}',
        color='#ee6677', fontsize=9, va='bottom')

for spine in ('top', 'right'):
    ax.spines[spine].set_visible(False)
ax.set_ylabel('Mean total bill ($)')
ax.set_ylim(0)
ax.set_title('Saturday bills are well above the weekly average —\n'
             'adding the reference line saves the reader a calculation (n=244)', loc='left')

plt.tight_layout()
plt.savefig('tips_reference_line.png', dpi=300, bbox_inches='tight')
plt.show()

print("""
Why 'maximise data-ink' is a heuristic, not a rule:
The reference line adds ink, but it removes a mental calculation.
Without it, the reader must estimate the mean from the four bars,
hold it in working memory, then compare. That is extraneous load.
The line trades a small amount of extra ink for a reduction in
the cognitive work the reader must do. The data-ink ratio falls;
the comprehension speed rises. Tufte's ratio is a useful pressure
toward simplicity, not an absolute rule.
""")